# In-Memory Vector Store: Lightweight Prototyping (Lecture 34)

`InMemoryVectorStore` (`langchain_core.vectorstores.InMemoryVectorStore`) is a lightweight, zero-dependency in-process vector store.
Ideal for:
- Unit tests and automated integration testing.
- Temporary session caching.
- Prototyping before deploying persistent databases like ChromaDB or Pinecone.

## 1. Setup & Corpus Ingestion
Populate InMemoryVectorStore with multi-domain documents.

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_documents = [
    Document(page_content="I had delicious chocolate chip pancakes and scrambled eggs for breakfast.", metadata={"topic": "food", "urgency": "low"}),
    Document(page_content="The weather forecast predicts heavy thunderstorms and high humidity tomorrow.", metadata={"topic": "weather", "urgency": "medium"}),
    Document(page_content="CRITICAL ALERT: Database cluster CPU utilization has exceeded 95% threshold.", metadata={"topic": "infrastructure", "urgency": "high"}),
    Document(page_content="Bank fraud and financial embezzlement carry severe criminal penalties under federal law.", metadata={"topic": "law", "urgency": "high"}),
]

vector_store = InMemoryVectorStore.from_documents(sample_documents, embeddings)
print("InMemoryVectorStore populated successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

InMemoryVectorStore populated successfully.


## 2. Similarity Search & Functional Metadata Filtering
Perform semantic search and apply metadata filter predicates.

In [2]:
query = "How is the weather forecast looking?"
results = vector_store.similarity_search(query, k=1)
print(f"Query: '{query}'")
print(f"Result: {results[0].page_content} (topic: {results[0].metadata['topic']})\n")

assert results[0].metadata["topic"] == "weather", "Top result should be weather"

# Functional metadata filtering
high_urgency_filter = lambda doc: doc.metadata.get("urgency") == "high"
filtered_results = vector_store.similarity_search("alert system issue", k=2, filter=high_urgency_filter)

print("Filtered Results (urgency=high):")
for doc in filtered_results:
    print(f"- {doc.page_content} [urgency: {doc.metadata['urgency']}]")
    assert doc.metadata["urgency"] == "high", "Filtering failed"

print("Similarity search and filtering assertions passed.")

Query: 'How is the weather forecast looking?'
Result: The weather forecast predicts heavy thunderstorms and high humidity tomorrow. (topic: weather)

Filtered Results (urgency=high):
- CRITICAL ALERT: Database cluster CPU utilization has exceeded 95% threshold. [urgency: high]
- Bank fraud and financial embezzlement carry severe criminal penalties under federal law. [urgency: high]
Similarity search and filtering assertions passed.
